# Warli DCGAN vs WGAN-GP — Final Multi-Seed Colab Pipeline

This notebook trains DCGAN and WGAN-GP independently using seeds **42, 123, and 2024**.  
The primary comparison is made at the common, pre-specified **epoch-100 checkpoint**.  
Checkpoint ranking is saved only as a secondary training-progression analysis.

Outputs include:
- per-seed checkpoint metrics;
- epoch-level training histories;
- epoch-50 and epoch-100 fixed-noise sample grids;
- foreground-mask diagnostics;
- SSIM nearest-neighbour comparison figures;
- run-level and architecture-level CSV summaries; and
- paired exploratory statistical comparisons.


## 1. Install dependencies

In [ ]:
!pip install -q torchmetrics torch-fidelity lpips scikit-image


## 2. Imports

In [ ]:
from __future__ import annotations

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from skimage.metrics import structural_similarity
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
import lpips


## 3. Configuration — edit these before running

In [ ]:
# --- Paths: edit these for your Google Drive ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Google Drive mount skipped.")

DATA_DIR = "/content/drive/MyDrive/Warli_Project/Dataset/train"  # ImageFolder root
OUTPUT_DIR = "/content/drive/MyDrive/Warli_Project/MultiSeed"

# --- Final training configuration ---
FULL_EPOCHS = 100
CHECKPOINT_EPOCHS = [25, 50, 75, 100]
SEEDS = [42, 123, 2024]

# --- Smoke test: run once, then change to False and use Runtime > Restart and run all ---
QUICK_TEST = True

# --- Nearest-neighbour options ---
SKIP_NEAREST_NEIGHBOUR = False
NN_SUBSAMPLE = None  # Keep None for the paper: compare with all 998 training images

# --- Fixed constants ---
LATENT_DIM = 100
IMAGE_SIZE = 64
IMAGE_CHANNELS = 1
BATCH_SIZE = 64
EVALUATION_SEED = 2026
REAL_REFERENCE_COUNT = 500
SSIM_GENERATED_COUNT = 100
SSIM_REFERENCES_PER_IMAGE = 5
LPIPS_PAIR_COUNT = 500
NN_GENERATED_COUNT = 100
PRIMARY_EPOCH = 100
SYMMETRY_THRESHOLD = 0.55

if QUICK_TEST:
    EPOCHS = 2
    CHECKPOINT_EVERY = 1
    ACTIVE_SEEDS = [42]
    RUN_NAME = "quick_test"
    PRIMARY_EPOCH_ACTIVE = 2
    print(">>> QUICK TEST MODE: 2 epochs, seed 42 only.")
    print(">>> After success: set QUICK_TEST=False, restart runtime, and run all.")
    print()
else:
    EPOCHS = FULL_EPOCHS
    CHECKPOINT_EVERY = 25
    ACTIVE_SEEDS = SEEDS
    RUN_NAME = "full_3seed"
    PRIMARY_EPOCH_ACTIVE = PRIMARY_EPOCH
    print(">>> FULL MODE: 100 epochs for seeds 42, 123, and 2024.")
    print()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

candidate_folders = []

for folder in drive_root.rglob("*"):
    if folder.is_dir():
        try:
            image_count = sum(
                1 for file in folder.iterdir()
                if file.is_file() and file.suffix.lower() in image_extensions
            )
            if image_count > 0:
                candidate_folders.append((str(folder), image_count))
        except (PermissionError, OSError):
            pass

for folder, count in sorted(
    candidate_folders,
    key=lambda item: item[1],
    reverse=True
)[:30]:
    print(f"{count:5d} images : {folder}")

## 4. Model architectures

**Verify these against your original notebook before trusting results (see note at the top).** Reconstructed from the paper's Section 3.2 / 3.3 specification.

In [ ]:
class DCGANGenerator(nn.Module):
    """Feature-map progression 100 -> 512 -> 256 -> 128 -> 64 -> 1."""

    def __init__(self, latent_dim: int = LATENT_DIM, feature_maps: int = 64):
        super().__init__()
        fm = feature_maps
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, fm * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(fm * 8, fm * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(fm * 4, fm * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(fm * 2, fm, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(fm, IMAGE_CHANNELS, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class DCGANDiscriminator(nn.Module):
    """Feature-map progression 1 -> 64 -> 128 -> 256 -> 512 -> 1."""

    def __init__(self, feature_maps: int = 64):
        super().__init__()
        fm = feature_maps
        self.net = nn.Sequential(
            nn.Conv2d(IMAGE_CHANNELS, fm, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm, fm * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 2, fm * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 4, fm * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)


class WGANGenerator(DCGANGenerator):
    """Identical architecture to the DCGAN generator (per paper, Section 3.3)."""
    pass


class WGANCritic(nn.Module):
    """Same channel progression as the discriminator, instance norm, no sigmoid."""

    def __init__(self, feature_maps: int = 64):
        super().__init__()
        fm = feature_maps
        self.net = nn.Sequential(
            nn.Conv2d(IMAGE_CHANNELS, fm, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm, fm * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(fm * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 2, fm * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(fm * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 4, fm * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(fm * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fm * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def initialize_weights(module: nn.Module) -> None:
    classname = module.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname or "InstanceNorm" in classname:
        if module.weight is not None:
            nn.init.normal_(module.weight.data, 1.0, 0.02)
        if module.bias is not None:
            nn.init.constant_(module.bias.data, 0)


def calculate_gradient_penalty(critic, real_images, fake_images, device=DEVICE):
    batch_size = real_images.size(0)
    epsilon = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (epsilon * real_images + (1 - epsilon) * fake_images).requires_grad_(True)
    scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    return ((gradient_norm - 1) ** 2).mean()


## 5. Data loading

In [ ]:
def build_dataloader(data_dir: str):
    transform = transforms.Compose(
        [
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),  # -> approx [-1, 1]
        ]
    )
    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    return loader, dataset


def sample_real_reference_images(dataset, count: int, seed: int) -> torch.Tensor:
    """Fixed pool of real images used for FID / SSIM / nearest-neighbour eval.
    Sampled once with a fixed seed, independent of the training seed."""
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(count, len(dataset)), replace=False)
    images = torch.stack([dataset[i][0] for i in indices])
    return images


def load_full_training_set_as_numpy(dataset) -> np.ndarray:
    """All images in the dataset, as [0,1]-range numpy arrays, for the
    nearest-neighbour SSIM comparison pool (paper compares against all 998,
    not just the 500-image evaluation subset)."""
    images = []
    for i in range(len(dataset)):
        img_tensor, _ = dataset[i]
        img = (img_tensor.squeeze(0).numpy() + 1.0) / 2.0  # [-1,1] -> [0,1]
        images.append(img)
    return np.stack(images)


## 6. Training loops

In [ ]:
def train_dcgan_one_seed(loader, seed, output_dir, total_epochs=100, checkpoint_every=25):
    set_seed(seed)
    run_dir = Path(output_dir) / f"DCGAN_seed{seed}"
    checkpoint_dir = run_dir / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    generator = DCGANGenerator().to(DEVICE)
    discriminator = DCGANDiscriminator().to(DEVICE)
    generator.apply(initialize_weights)
    discriminator.apply(initialize_weights)

    criterion = nn.BCEWithLogitsLoss()
    optimizer_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))
    history = []

    print()
    print(f"=== DCGAN | seed {seed} ===")
    for epoch in range(1, total_epochs + 1):
        generator.train(); discriminator.train()
        epoch_start = time.time()
        running_g = running_d = 0.0

        for real_images, _ in loader:
            real_images = real_images.to(DEVICE, non_blocking=True)
            batch_size = real_images.size(0)
            real_targets = torch.full((batch_size,), 0.9, device=DEVICE)
            fake_targets = torch.zeros(batch_size, device=DEVICE)
            gen_targets = torch.ones(batch_size, device=DEVICE)

            optimizer_D.zero_grad(set_to_none=True)
            loss_d_real = criterion(discriminator(real_images), real_targets)
            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
            fake_images = generator(noise)
            loss_d_fake = criterion(discriminator(fake_images.detach()), fake_targets)
            loss_d = 0.5 * (loss_d_real + loss_d_fake)
            loss_d.backward()
            nn.utils.clip_grad_norm_(discriminator.parameters(), 5.0)
            optimizer_D.step()

            optimizer_G.zero_grad(set_to_none=True)
            loss_g = criterion(discriminator(fake_images), gen_targets)
            loss_g.backward()
            nn.utils.clip_grad_norm_(generator.parameters(), 5.0)
            optimizer_G.step()
            running_g += loss_g.item(); running_d += loss_d.item()

        g_mean = running_g / len(loader)
        d_mean = running_d / len(loader)
        history.append({"epoch": epoch, "generator_loss": g_mean, "discriminator_loss": d_mean})

        if epoch % 10 == 0 or epoch == total_epochs or QUICK_TEST:
            print(f"  epoch {epoch:03d}/{total_epochs} | G {g_mean:.4f} | D {d_mean:.4f} | {time.time()-epoch_start:.1f}s")

        if epoch % checkpoint_every == 0 or epoch == total_epochs:
            torch.save({"epoch": epoch, "generator_state_dict": generator.state_dict()},
                       checkpoint_dir / f"dcgan_seed{seed}_epoch_{epoch:03d}.pth")

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / f"dcgan_seed{seed}_training_history.csv", index=False)
    return checkpoint_dir, history_df


def train_wgan_gp_one_seed(loader, seed, output_dir, total_epochs=100, checkpoint_every=25,
                           critic_steps=3, gradient_penalty_weight=5.0):
    set_seed(seed)
    run_dir = Path(output_dir) / f"WGAN_GP_seed{seed}"
    checkpoint_dir = run_dir / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    generator = WGANGenerator().to(DEVICE)
    critic = WGANCritic().to(DEVICE)
    generator.apply(initialize_weights); critic.apply(initialize_weights)
    optimizer_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.0, 0.9))
    optimizer_C = optim.Adam(critic.parameters(), lr=1e-4, betas=(0.0, 0.9))
    history = []

    print()
    print(f"=== WGAN-GP | seed {seed} ===")
    for epoch in range(1, total_epochs + 1):
        generator.train(); critic.train()
        epoch_start = time.time()
        running_g = running_c = running_gp = running_w = 0.0
        updates = 0

        for real_images, _ in loader:
            real_images = real_images.to(DEVICE, non_blocking=True)
            batch_size = real_images.size(0)

            for _ in range(critic_steps):
                optimizer_C.zero_grad(set_to_none=True)
                noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
                with torch.no_grad():
                    fake_images = generator(noise)
                real_scores = critic(real_images)
                fake_scores = critic(fake_images)
                gp = calculate_gradient_penalty(critic, real_images, fake_images)
                wasserstein_sep = real_scores.mean() - fake_scores.mean()
                critic_loss = -wasserstein_sep + gradient_penalty_weight * gp
                critic_loss.backward()
                nn.utils.clip_grad_norm_(critic.parameters(), 10.0)
                optimizer_C.step()

            optimizer_G.zero_grad(set_to_none=True)
            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
            generated = generator(noise)
            generator_loss = -critic(generated).mean()
            generator_loss.backward()
            nn.utils.clip_grad_norm_(generator.parameters(), 10.0)
            optimizer_G.step()

            running_g += generator_loss.item(); running_c += critic_loss.item()
            running_gp += gp.item(); running_w += wasserstein_sep.item(); updates += 1

        values = {
            "epoch": epoch,
            "generator_loss": running_g / updates,
            "critic_loss": running_c / updates,
            "gradient_penalty": running_gp / updates,
            "wasserstein_separation": running_w / updates,
        }
        history.append(values)
        if epoch % 10 == 0 or epoch == total_epochs or QUICK_TEST:
            print(f"  epoch {epoch:03d}/{total_epochs} | G {values['generator_loss']:.4f} | C {values['critic_loss']:.4f} | GP {values['gradient_penalty']:.4f} | W {values['wasserstein_separation']:.4f} | {time.time()-epoch_start:.1f}s")

        if epoch % checkpoint_every == 0 or epoch == total_epochs:
            torch.save({"epoch": epoch, "generator_state_dict": generator.state_dict()},
                       checkpoint_dir / f"wgan_gp_seed{seed}_epoch_{epoch:03d}.pth")

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / f"wgan_gp_seed{seed}_training_history.csv", index=False)
    return checkpoint_dir, history_df


## 7. Evaluation functions

In [ ]:
def generate_evaluation_images(generator, noise: torch.Tensor) -> torch.Tensor:
    generator.eval()
    with torch.no_grad():
        images = generator(noise.to(DEVICE))
    return ((images + 1.0) / 2.0).clamp(0, 1).cpu()


def compute_fid(real_images_01, generated_images_01):
    metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    metric.update(real_images_01.repeat(1, 3, 1, 1).to(DEVICE), real=True)
    metric.update(generated_images_01.repeat(1, 3, 1, 1).to(DEVICE), real=False)
    value = float(metric.compute().cpu())
    del metric
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return value


def compute_ssim_protocol(generated_numpy, real_numpy, seed=42):
    rng = np.random.default_rng(seed)
    n = min(SSIM_GENERATED_COUNT, len(generated_numpy))
    per_image_means = []
    for i in range(n):
        refs = rng.choice(len(real_numpy), size=SSIM_REFERENCES_PER_IMAGE, replace=False)
        scores = [structural_similarity(generated_numpy[i], real_numpy[j], data_range=1.0) for j in refs]
        per_image_means.append(np.mean(scores))
    values = np.asarray(per_image_means)
    return float(values.mean()), float(values.std(ddof=1))


def axial_symmetry_score(image, threshold=SYMMETRY_THRESHOLD):
    mirrored = image[:, ::-1]
    joint = (image >= threshold) | (mirrored >= threshold)
    if joint.sum() == 0: return 0.0
    discrepancy = np.abs(image[joint] - mirrored[joint]).mean()
    return float(np.clip(1.0 - discrepancy, 0.0, 1.0))


def compute_lpips_diversity(generated_images_01, loss_fn, seed=42):
    rng = np.random.default_rng(seed)
    n = generated_images_01.size(0)
    pairs = [rng.choice(n, size=2, replace=False) for _ in range(LPIPS_PAIR_COUNT)]
    rgb = (generated_images_01 * 2.0 - 1.0).repeat(1, 3, 1, 1)
    distances = []
    with torch.no_grad():
        for a, b in pairs:
            distances.append(loss_fn(rgb[a:a+1].to(DEVICE), rgb[b:b+1].to(DEVICE)).item())
    values = np.asarray(distances)
    return float(values.mean()), float(values.std(ddof=1))


def nearest_neighbour_ssim(generated_numpy, training_numpy, seed=42, n_generated=NN_GENERATED_COUNT):
    rng = np.random.default_rng(seed)
    selected = rng.choice(len(generated_numpy), size=min(n_generated, len(generated_numpy)), replace=False)
    scores, train_indices = [], []
    for gen_idx in selected:
        current = np.asarray([structural_similarity(generated_numpy[gen_idx], tr, data_range=1.0)
                              for tr in training_numpy])
        best_idx = int(np.argmax(current))
        scores.append(float(current[best_idx])); train_indices.append(best_idx)
    scores = np.asarray(scores)
    summary = {"mean": float(scores.mean()), "std": float(scores.std(ddof=1)),
               "min": float(scores.min()), "max": float(scores.max())}
    matches = pd.DataFrame({"generated_index": selected, "training_index": train_indices, "ssim": scores})
    return summary, matches


def save_sample_grid(images, path, nrow=10):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    save_image(images[:100], path, nrow=nrow, normalize=False, padding=2, pad_value=1.0)


def save_training_plot(history_df, architecture, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if architecture == "DCGAN":
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(history_df.epoch, history_df.generator_loss); axes[0].set_title("Generator loss")
        axes[1].plot(history_df.epoch, history_df.discriminator_loss); axes[1].set_title("Discriminator loss")
    else:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        columns = ["generator_loss", "critic_loss", "gradient_penalty", "wasserstein_separation"]
        titles = ["Generator loss", "Critic loss", "Gradient penalty", "Wasserstein separation"]
        for ax, col, title in zip(axes.ravel(), columns, titles):
            ax.plot(history_df.epoch, history_df[col]); ax.set_title(title)
    for ax in np.asarray(axes).ravel(): ax.set_xlabel("Epoch"); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def save_symmetry_diagnostic(images_01, path, count=8):
    images = images_01[:count].squeeze(1).numpy()
    fig, axes = plt.subplots(2, count, figsize=(2*count, 4))
    for i, image in enumerate(images):
        axes[0, i].imshow(image, cmap="gray", vmin=0, vmax=1); axes[0, i].axis("off")
        axes[1, i].imshow(image >= SYMMETRY_THRESHOLD, cmap="gray", vmin=0, vmax=1); axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Image"); axes[1, 0].set_ylabel("Mask")
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def save_nn_figure(generated_numpy, training_numpy, matches, path, count=5):
    shown = matches.sort_values("generated_index").iloc[:count]
    fig, axes = plt.subplots(count, 2, figsize=(5, 2.2*count))
    for row, (_, match) in enumerate(shown.iterrows()):
        gi, ti = int(match.generated_index), int(match.training_index)
        axes[row, 0].imshow(generated_numpy[gi], cmap="gray", vmin=0, vmax=1)
        axes[row, 1].imshow(training_numpy[ti], cmap="gray", vmin=0, vmax=1)
        axes[row, 0].set_ylabel(f"SSIM {match.ssim:.3f}")
        axes[row, 0].axis("off"); axes[row, 1].axis("off")
    axes[0, 0].set_title("Generated"); axes[0, 1].set_title("Nearest real")
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def evaluate_all_checkpoints(checkpoint_dir, model_class, real_images_01, real_numpy,
                             fixed_noise, architecture_name):
    rows = []
    for path in sorted(Path(checkpoint_dir).glob("*.pth")):
        checkpoint = torch.load(path, map_location=DEVICE)
        model = model_class().to(DEVICE); model.load_state_dict(checkpoint["generator_state_dict"])
        generated = generate_evaluation_images(model, fixed_noise)
        generated_np = generated.squeeze(1).numpy()
        fid = compute_fid(real_images_01, generated)
        sm, ss = compute_ssim_protocol(generated_np, real_numpy)
        syms = np.asarray([axial_symmetry_score(img) for img in generated_np])
        rows.append({"architecture": architecture_name, "epoch": checkpoint["epoch"],
                     "checkpoint": path.name, "fid": fid, "ssim_mean": sm, "ssim_std": ss,
                     "symmetry_mean": float(syms.mean()), "symmetry_std": float(syms.std(ddof=1))})
        print(f"    epoch {checkpoint['epoch']:03d} | FID {fid:.2f} | SSIM {sm:.4f} | Symmetry {syms.mean():.4f}")
        del model
    return pd.DataFrame(rows).sort_values("epoch")


def add_checkpoint_ranks(df):
    out = df.copy()
    out["fid_rank"] = out.fid.rank(method="min", ascending=True)
    out["ssim_rank"] = out.ssim_mean.rank(method="min", ascending=False)
    out["symmetry_rank"] = out.symmetry_mean.rank(method="min", ascending=False)
    out["combined_rank"] = out[["fid_rank", "ssim_rank", "symmetry_rank"]].sum(axis=1)
    return out.sort_values(["combined_rank", "fid"])


## 8. Setup — load data, build fixed evaluation pools

In [ ]:
base_output_dir = Path(OUTPUT_DIR)
output_dir = base_output_dir / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

loader, dataset = build_dataloader(DATA_DIR)
print(f"Dataset size: {len(dataset)} images")
if not QUICK_TEST and len(dataset) != 998:
    print(f"WARNING: paper states 998 images, but ImageFolder loaded {len(dataset)}.")

real_images_01 = (sample_real_reference_images(dataset, REAL_REFERENCE_COUNT, EVALUATION_SEED) + 1.0) / 2.0
real_numpy = real_images_01.squeeze(1).numpy()
full_training_numpy = load_full_training_set_as_numpy(dataset)

if NN_SUBSAMPLE:
    rng = np.random.default_rng(EVALUATION_SEED)
    idx = rng.choice(len(full_training_numpy), size=min(NN_SUBSAMPLE, len(full_training_numpy)), replace=False)
    nn_reference_pool = full_training_numpy[idx]
    print(f"Nearest-neighbour pool subsampled to {len(nn_reference_pool)} images (exploratory only).")
else:
    nn_reference_pool = full_training_numpy

noise_generator = torch.Generator().manual_seed(EVALUATION_SEED)
fixed_noise = torch.randn(REAL_REFERENCE_COUNT, LATENT_DIM, 1, 1, generator=noise_generator)
lpips_loss_fn = lpips.LPIPS(net="alex").to(DEVICE).eval()
print(f"Outputs: {output_dir}")


## 9. Main loop — train + evaluate DCGAN and WGAN-GP across all seeds

This is the long-running cell. Progress (and partial results) are saved after every seed to `multi_seed_results_partial.csv`, so a runtime disconnect doesn't lose completed work — just re-run from where it left off by adjusting `SEEDS` above.

In [ ]:
all_results = []

def run_architecture(architecture, seed, trainer, model_class):
    ckpt_dir, history_df = trainer(loader, seed, output_dir, EPOCHS, CHECKPOINT_EVERY)
    run_dir = ckpt_dir.parent
    save_training_plot(history_df, architecture, run_dir / f"{architecture.lower().replace('-', '_')}_training_dynamics.png")

    evaluation = evaluate_all_checkpoints(ckpt_dir, model_class, real_images_01, real_numpy,
                                          fixed_noise, architecture)
    ranked = add_checkpoint_ranks(evaluation)
    evaluation.to_csv(run_dir / "checkpoint_metrics.csv", index=False)
    ranked.to_csv(run_dir / "checkpoint_metrics_with_ranks.csv", index=False)

    # Save fixed-noise grids at every available checkpoint; paper can use epochs 50 and 100.
    for _, checkpoint_row in evaluation.iterrows():
        checkpoint = torch.load(ckpt_dir / checkpoint_row.checkpoint, map_location=DEVICE)
        model = model_class().to(DEVICE); model.load_state_dict(checkpoint["generator_state_dict"])
        images = generate_evaluation_images(model, fixed_noise)
        save_sample_grid(images, run_dir / f"generated_samples_epoch_{int(checkpoint_row.epoch):03d}.png")
        del model

    # Primary comparison: identical pre-specified epoch for both architectures.
    primary_rows = evaluation.loc[evaluation.epoch == PRIMARY_EPOCH_ACTIVE]
    if primary_rows.empty:
        raise RuntimeError(f"Epoch {PRIMARY_EPOCH_ACTIVE} checkpoint is missing for {architecture}, seed {seed}.")
    primary = primary_rows.iloc[0]
    checkpoint = torch.load(ckpt_dir / primary.checkpoint, map_location=DEVICE)
    model = model_class().to(DEVICE); model.load_state_dict(checkpoint["generator_state_dict"])
    images = generate_evaluation_images(model, fixed_noise)
    generated_np = images.squeeze(1).numpy()

    lpips_mean, lpips_std = compute_lpips_diversity(images, lpips_loss_fn)
    sym_values = np.asarray([axial_symmetry_score(img) for img in generated_np])
    save_symmetry_diagnostic(images, run_dir / "symmetry_mask_diagnostic.png")

    nn_summary = {"mean": np.nan, "std": np.nan, "min": np.nan, "max": np.nan}
    if not SKIP_NEAREST_NEIGHBOUR:
        nn_summary, matches = nearest_neighbour_ssim(generated_np, nn_reference_pool, seed=42)
        matches.to_csv(run_dir / "nearest_neighbour_matches.csv", index=False)
        save_nn_figure(generated_np, nn_reference_pool, matches, run_dir / "nearest_neighbours.png")

    result = {
        "architecture": architecture, "seed": seed, "primary_epoch": PRIMARY_EPOCH_ACTIVE,
        "fid": float(primary.fid), "ssim": float(primary.ssim_mean),
        "symmetry": float(primary.symmetry_mean), "lpips_diversity": lpips_mean,
        "lpips_pair_sd": lpips_std, "nn_ssim_mean": nn_summary["mean"],
        "nn_ssim_within_run_sd": nn_summary["std"], "nn_ssim_min": nn_summary["min"],
        "nn_ssim_max": nn_summary["max"], "rank_selected_epoch_secondary": int(ranked.iloc[0].epoch),
    }
    del model
    return result


for seed in ACTIVE_SEEDS:
    all_results.append(run_architecture("DCGAN", seed, train_dcgan_one_seed, DCGANGenerator))
    pd.DataFrame(all_results).to_csv(output_dir / "multi_seed_results_partial.csv", index=False)
    all_results.append(run_architecture("WGAN-GP", seed, train_wgan_gp_one_seed, WGANGenerator))
    pd.DataFrame(all_results).to_csv(output_dir / "multi_seed_results_partial.csv", index=False)

results_df = pd.DataFrame(all_results).sort_values(["architecture", "seed"])
results_df.to_csv(output_dir / "multi_seed_results_full.csv", index=False)
print()
print("=== Primary common-checkpoint results ===")
display(results_df)


## 10. Aggregate — mean ± SD per architecture (paste into paper)

In [ ]:
metrics = ["fid", "ssim", "symmetry", "lpips_diversity", "nn_ssim_mean", "nn_ssim_max"]
summary_rows = []
for architecture, subset in results_df.groupby("architecture"):
    row = {"Architecture": architecture, "N seeds": len(subset), "Primary epoch": PRIMARY_EPOCH_ACTIVE}
    for metric in metrics:
        values = subset[metric].dropna()
        if len(values):
            sd = values.std(ddof=1) if len(values) > 1 else np.nan
            row[f"{metric} mean"] = values.mean()
            row[f"{metric} between-seed SD"] = sd
            row[f"{metric} formatted"] = f"{values.mean():.4f} +/- {sd:.4f}" if len(values) > 1 else f"{values.mean():.4f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(output_dir / "multi_seed_summary_table.csv", index=False)
print("=== Architecture-level mean and between-seed SD ===")
display(summary_df)


## 11. Rough DCGAN vs WGAN-GP comparison

**n=3 per arm gives a t-test very low power — treat this as a sanity check, report the effect size (Cohen's d) alongside the p-value in the paper rather than leaning on the p-value alone.**

In [ ]:
comparison_rows = []
if set(results_df.architecture) == {"DCGAN", "WGAN-GP"}:
    print("=== Paired exploratory comparisons (same seeds; low power when n=3) ===")
    for metric in ["fid", "ssim", "symmetry", "lpips_diversity", "nn_ssim_mean"]:
        paired = results_df.pivot(index="seed", columns="architecture", values=metric).dropna().sort_index()
        a, b = paired["DCGAN"].to_numpy(), paired["WGAN-GP"].to_numpy()
        if len(a) >= 2:
            t_stat, p_value = stats.ttest_rel(a, b)
            differences = a - b
            paired_d = differences.mean() / differences.std(ddof=1) if differences.std(ddof=1) > 0 else np.nan
        else:
            t_stat = p_value = paired_d = np.nan
        comparison_rows.append({"metric": metric, "n_pairs": len(a), "DCGAN_mean": a.mean(),
                                "WGAN_GP_mean": b.mean(), "mean_paired_difference_D_minus_W": (a-b).mean(),
                                "paired_t": t_stat, "p_value_exploratory": p_value, "paired_Cohens_dz": paired_d})
    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(output_dir / "paired_exploratory_comparisons.csv", index=False)
    display(comparison_df)
else:
    print("Both architectures are not yet available.")

print()
print(f"All outputs written to: {output_dir}")
if QUICK_TEST:
    print("QUICK TEST FINISHED. Now set QUICK_TEST=False, restart the runtime, and run all cells.")
else:
    print("FULL THREE-SEED RUN FINISHED. Preserve the full_3seed folder before updating the paper.")
